# Week 2 Day 3 — LangGraph: Stateful, Multi-Step & Cyclical Agent Workflows

**Workflow chosen:** a *Research Assistant* agent that plans a research approach, retrieves
information (reusing the Day 2 search tool), drafts an answer, critiques its own draft, loops
back to revise when quality is too low, pauses for human approval before "publishing" the final
answer (the risky action), and persists its state across runs with a checkpointer.

This notebook builds the graph up incrementally, exactly following the five tasks in the brief.


## Setup

We use `langgraph` for the graph runtime and reuse the idea of tools from Day 2 (LangChain).
A `GEMINI_API_KEY` (or `GOOGLE_API_KEY`) environment variable is picked up automatically if
present, and `call_llm()` routes to Google's Gemini API; if no key is set it falls back to a
small deterministic simulator so the whole notebook still runs end-to-end for grading/demo
purposes. Swapping in a real key requires no other code changes.



In [1]:
import os
import json
import random
from typing import TypedDict, List, Optional

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

random.seed(7)


_gemini_key = os.environ.get("")
USE_REAL_LLM = bool(_gemini_key)
print(f"Using real Gemini API: {USE_REAL_LLM}")

if USE_REAL_LLM:
    from google import genai
    _client = genai.Client(api_key=_gemini_key)

    def call_llm(prompt: str, system: str = "") -> str:
        resp = _client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=prompt,
            config={"system_instruction": system} if system else None,
        )
        return resp.text
else:
    # Deterministic stand-in so the notebook is fully runnable without an API key.
    # Swap this out for the branch above in a real deployment.
    def call_llm(prompt: str, system: str = "") -> str:
        if "Write a short research plan" in prompt:
            return "1) Clarify the question 2) Gather 2-3 supporting facts 3) Synthesize a concise answer"
        if "Draft an answer" in prompt:
            attempt = prompt.count("Revision")
            base = "LangGraph models agents as explicit graphs of nodes and edges over a shared state object."
            if attempt >= 1:
                base += " Compared to a plain loop, this makes branching, cycles, and pausing first-class concepts."
            return base
        if "Critique the following draft" in prompt:
            draft = prompt.split(":", 1)[-1]
            score = min(0.95, 0.55 + 0.2 * prompt.count("Revision"))
            verdict = "Solid, but add a concrete example." if score < 0.8 else "Clear and sufficiently detailed."
            return json.dumps({"score": round(score, 2), "feedback": verdict})
        return "OK"

print("call_llm ready")


Using real Gemini API: False
call_llm ready


## Task 1 — Graph Concepts & State Design

**Core LangGraph building blocks**

- **`StateGraph`** — the graph builder. You register nodes and edges on it, then `.compile()`
  it into a runnable graph. It's generic over a state schema, so LangGraph knows how to merge
  each node's return value into the shared state.
- **Nodes** — plain Python callables (or LangChain runnables) of the form
  `def node(state: State) -> dict`. Each node reads whatever fields it needs from `state` and
  returns a **partial update** (only the keys it changed); LangGraph merges that into the
  overall state.
- **Edges** — unconditional transitions, `graph.add_edge("a", "b")`, meaning "always go from
  node `a` to node `b`".
- **Conditional edges** — `graph.add_conditional_edges("a", router_fn, {"x": "b", "y": "c"})`.
  `router_fn(state)` inspects the state and returns a label; LangGraph follows the edge that
  matches. This is how branching *and* cycles (loop-backs) are expressed — a conditional edge
  is allowed to point back to a node that already ran.
- **State** — a single `TypedDict` (or Pydantic model) shared by every node. It's the only
  channel nodes communicate through, which is what makes the whole run inspectable and
  resumable: at any point "the state" *is* the full memory of the run.

**State schema for the Research Assistant workflow:**


In [2]:
class AgentState(TypedDict):
    query: str                     # the user's research question
    plan: str                      # research plan produced once, up front
    search_results: List[str]      # retrieved snippets (Day 2 search tool)
    draft: str                     # current draft answer
    critique: str                  # feedback text from the critique node
    quality_score: float           # 0-1 score assigned by the critique node
    revision_count: int            # how many times we've looped back to generate
    max_revisions: int             # safety cap on the self-correction loop
    approved: Optional[bool]       # human decision at the interrupt point
    final_answer: str              # the answer that got published
    log: List[str]                 # human-readable trace of every node that ran


**Graph we are about to build (final version, after Tasks 2-5):**

```
        START
          |
          v
       [ plan ]
          |
          v
      [ retrieve ]              <- Day 2 search tool
          |
          v
   +-> [ generate ] --------------------+
   |      |                             |
   |      v                             |
   |  [ critique ]                      |
   |      |                             |
   |      | quality < threshold          |
   |      | AND revisions < max          |
   +------+                             |
          | quality >= threshold         |
          | OR revisions == max          |
          v                             |
     [ human_review ]  <-- interrupt_before, needs approval
          |
     approved? --no--> [ reject ] --> END
          |
         yes
          v
      [ publish ]  (the "risky" action)
          |
          v
         END
```

Task 2 builds the top `plan -> retrieve -> generate -> format` spine as a plain linear graph.
Task 3 turns `generate -> critique` into a real self-correction loop with a conditional edge.
Task 4 adds the `human_review` interrupt in front of `publish`. Task 5 wraps the whole thing
with a checkpointer for persistence and time-travel debugging.


## Task 2 — Build a Linear Graph

A minimal 4-node linear graph: `plan -> retrieve -> generate -> format`. `retrieve` reuses the
same idea as the Day 2 `search_web` tool (kept as a small standalone function here so this
notebook has no dependency on the Day 2 file).


In [3]:
def search_web(query: str) -> List[str]:
    """Stand-in for the Day 2 `search_web` tool: returns a couple of fake snippets."""
    return [
        f"Snippet 1 about '{query}': LangGraph builds agents as graphs over shared state.",
        f"Snippet 2 about '{query}': Conditional edges enable branching and cycles.",
    ]


def plan_node(state: AgentState) -> dict:
    plan = call_llm(f"Write a short research plan for: {state["query"]}")
    return {"plan": plan, "log": state.get("log", []) + ["plan"]}


def retrieve_node(state: AgentState) -> dict:
    results = search_web(state["query"])
    return {"search_results": results, "log": state["log"] + ["retrieve"]}


def generate_node(state: AgentState) -> dict:
    revision_note = f" (Revision {state["revision_count"]})" if state.get("revision_count") else ""
    prompt = (
        f"Draft an answer to '{state["query"]}' using: {state["search_results"]}."
        f" Plan: {state["plan"]}.{revision_note}"
    )
    draft = call_llm(prompt)
    return {"draft": draft, "log": state["log"] + ["generate"]}


def format_node(state: AgentState) -> dict:
    final = f"Q: {state["query"]}\nA: {state["draft"]}"
    return {"final_answer": final, "log": state["log"] + ["format"]}


In [4]:
linear_graph = StateGraph(AgentState)
linear_graph.add_node("plan", plan_node)
linear_graph.add_node("retrieve", retrieve_node)
linear_graph.add_node("generate", generate_node)
linear_graph.add_node("format", format_node)

linear_graph.add_edge(START, "plan")
linear_graph.add_edge("plan", "retrieve")
linear_graph.add_edge("retrieve", "generate")
linear_graph.add_edge("generate", "format")
linear_graph.add_edge("format", END)

linear_app = linear_graph.compile()

sample_input: AgentState = {
    "query": "What makes LangGraph different from a plain agent loop?",
    "plan": "", "search_results": [], "draft": "", "critique": "",
    "quality_score": 0.0, "revision_count": 0, "max_revisions": 2,
    "approved": None, "final_answer": "", "log": [],
}

print("--- Running linear graph, printing state after each node ---")
for update in linear_app.stream(sample_input, stream_mode="updates"):
    node_name, node_output = next(iter(update.items()))
    print(f"\n[after {node_name}]")
    for k, v in node_output.items():
        print(f"  {k}: {v}")


--- Running linear graph, printing state after each node ---

[after plan]
  plan: 1) Clarify the question 2) Gather 2-3 supporting facts 3) Synthesize a concise answer
  log: ['plan']

[after retrieve]
  search_results: ["Snippet 1 about 'What makes LangGraph different from a plain agent loop?': LangGraph builds agents as graphs over shared state.", "Snippet 2 about 'What makes LangGraph different from a plain agent loop?': Conditional edges enable branching and cycles."]
  log: ['plan', 'retrieve']

[after generate]
  draft: LangGraph models agents as explicit graphs of nodes and edges over a shared state object.
  log: ['plan', 'retrieve', 'generate']

[after format]
  final_answer: Q: What makes LangGraph different from a plain agent loop?
A: LangGraph models agents as explicit graphs of nodes and edges over a shared state object.
  log: ['plan', 'retrieve', 'generate', 'format']


## Task 3 — Add Conditional Edges & Cycles

We replace `format` with a `critique` node that scores the draft. A **conditional edge**
routes back to `generate` (looping) while the score is below a threshold *and* we haven't hit
`max_revisions`; otherwise it moves on to `format`. `revision_count` is the cycle guard.


In [5]:
QUALITY_THRESHOLD = 0.8

def critique_node(state: AgentState) -> dict:
    prompt = f"Critique the following draft: {state["draft"]}"
    raw = call_llm(prompt)
    try:
        parsed = json.loads(raw)
        score, feedback = parsed["score"], parsed["feedback"]
    except (json.JSONDecodeError, KeyError):
        score, feedback = 0.9, "Looks fine."
    log_line = f"critique(score={score})"
    return {
        "critique": feedback,
        "quality_score": score,
        "log": state["log"] + [log_line],
    }


def bump_revision_node(state: AgentState) -> dict:
    return {"revision_count": state["revision_count"] + 1, "log": state["log"] + ["revise"]}


def route_after_critique(state: AgentState) -> str:
    good_enough = state["quality_score"] >= QUALITY_THRESHOLD
    out_of_retries = state["revision_count"] >= state["max_revisions"]
    if good_enough or out_of_retries:
        return "finish"
    return "loop_back"


In [6]:
cyclical_graph = StateGraph(AgentState)
cyclical_graph.add_node("plan", plan_node)
cyclical_graph.add_node("retrieve", retrieve_node)
cyclical_graph.add_node("generate", generate_node)
cyclical_graph.add_node("critique", critique_node)
cyclical_graph.add_node("revise", bump_revision_node)
cyclical_graph.add_node("format", format_node)

cyclical_graph.add_edge(START, "plan")
cyclical_graph.add_edge("plan", "retrieve")
cyclical_graph.add_edge("retrieve", "generate")
cyclical_graph.add_edge("generate", "critique")
cyclical_graph.add_conditional_edges(
    "critique", route_after_critique, {"loop_back": "revise", "finish": "format"}
)
cyclical_graph.add_edge("revise", "generate")   # <- the actual loop-back
cyclical_graph.add_edge("format", END)

cyclical_app = cyclical_graph.compile()

print("--- Running graph with self-correction loop ---")
for update in cyclical_app.stream(sample_input, stream_mode="updates"):
    node_name, node_output = next(iter(update.items()))
    print(f"[after {node_name}] {node_output.get("log", [])[-1:]}  quality_score={node_output.get("quality_score")}")


--- Running graph with self-correction loop ---
[after plan] ['plan']  quality_score=None
[after retrieve] ['retrieve']  quality_score=None
[after generate] ['generate']  quality_score=None
[after critique] ['critique(score=0.55)']  quality_score=0.55
[after revise] ['revise']  quality_score=None
[after generate] ['generate']  quality_score=None
[after critique] ['critique(score=0.55)']  quality_score=0.55
[after revise] ['revise']  quality_score=None
[after generate] ['generate']  quality_score=None
[after critique] ['critique(score=0.55)']  quality_score=0.55
[after format] ['format']  quality_score=None


**Why is this loop-back pattern awkward in a plain `AgentExecutor` but natural in
LangGraph?** An `AgentExecutor` runs a single, implicit `think -> act -> observe` loop
controlled by the LLM deciding when to stop; there's no first-class way to say "go back to a
*specific earlier step* with updated state" — you'd have to fake it by stuffing critique
feedback back into the prompt and hoping the model re-plans correctly. In LangGraph the loop is
an explicit edge in the graph (`critique -> revise -> generate`) with its own guard
(`revision_count`), so the control flow — not the model's discretion — guarantees termination
and makes the cycle visible, testable, and debuggable independent of what the LLM decides.


## Task 4 — Human-in-the-Loop & Interrupts

The "risky" action here is **publishing** the final answer externally (imagine this were
"send the email" / "place the order"). We add a `human_review` node and use
`interrupt_before=["publish"]` so the graph pauses right before that action and waits for a
human decision, which we inject by updating state and resuming the run.


In [7]:
def human_review_node(state: AgentState) -> dict:
    # In a real product this would surface a UI/CLI prompt. Here it just logs that we
    # reached the checkpoint; the actual approve/reject value is supplied by the human
    # when they resume the run (see below).
    return {"log": state["log"] + ["human_review (awaiting approval)"]}


def publish_node(state: AgentState) -> dict:
    return {"final_answer": f"[PUBLISHED] Q: {state["query"]}\nA: {state["draft"]}",
            "log": state["log"] + ["publish"]}


def reject_node(state: AgentState) -> dict:
    return {"final_answer": "[NOT PUBLISHED - rejected by reviewer]",
            "log": state["log"] + ["reject"]}


def route_after_review(state: AgentState) -> str:
    return "publish" if state.get("approved") else "reject"


In [8]:
checkpointer = MemorySaver()

hitl_graph = StateGraph(AgentState)
hitl_graph.add_node("plan", plan_node)
hitl_graph.add_node("retrieve", retrieve_node)
hitl_graph.add_node("generate", generate_node)
hitl_graph.add_node("critique", critique_node)
hitl_graph.add_node("revise", bump_revision_node)
hitl_graph.add_node("human_review", human_review_node)
hitl_graph.add_node("publish", publish_node)
hitl_graph.add_node("reject", reject_node)

hitl_graph.add_edge(START, "plan")
hitl_graph.add_edge("plan", "retrieve")
hitl_graph.add_edge("retrieve", "generate")
hitl_graph.add_edge("generate", "critique")
hitl_graph.add_conditional_edges(
    "critique", route_after_critique, {"loop_back": "revise", "finish": "human_review"}
)
hitl_graph.add_edge("revise", "generate")
hitl_graph.add_conditional_edges(
    "human_review", route_after_review, {"publish": "publish", "reject": "reject"}
)
hitl_graph.add_edge("publish", END)
hitl_graph.add_edge("reject", END)

hitl_app = hitl_graph.compile(checkpointer=checkpointer, interrupt_before=["publish", "reject"])

thread = {"configurable": {"thread_id": "demo-thread-1"}}
result = hitl_app.invoke(sample_input, config=thread)
print("Paused. Next node(s) waiting to run:", hitl_app.get_state(thread).next)
print("Current approved value:", result.get("approved"))


Paused. Next node(s) waiting to run: ('reject',)
Current approved value: None


In [9]:
# Simulate a human REJECTING the risky action first.
hitl_app.update_state(thread, {"approved": False})
result = hitl_app.invoke(None, config=thread)   # resume from the interrupt
print("Resumed with rejection ->", result["final_answer"])
print("Trace:", result["log"])


Resumed with rejection -> [NOT PUBLISHED - rejected by reviewer]
Trace: ['plan', 'retrieve', 'generate', 'critique(score=0.55)', 'revise', 'generate', 'critique(score=0.55)', 'revise', 'generate', 'critique(score=0.55)', 'human_review (awaiting approval)', 'reject']


In [10]:
# Now simulate the human APPROVING on a fresh thread.
thread_2 = {"configurable": {"thread_id": "demo-thread-2"}}
hitl_app.invoke(sample_input, config=thread_2)
hitl_app.update_state(thread_2, {"approved": True})
result_2 = hitl_app.invoke(None, config=thread_2)
print("Resumed with approval ->", result_2["final_answer"])
print("Trace:", result_2["log"])


Resumed with approval -> [PUBLISHED] Q: What makes LangGraph different from a plain agent loop?
A: LangGraph models agents as explicit graphs of nodes and edges over a shared state object. Compared to a plain loop, this makes branching, cycles, and pausing first-class concepts.
Trace: ['plan', 'retrieve', 'generate', 'critique(score=0.55)', 'revise', 'generate', 'critique(score=0.55)', 'revise', 'generate', 'critique(score=0.55)', 'human_review (awaiting approval)', 'publish']


**When should a real product require human-in-the-loop, versus allow full autonomy?**
Require a human checkpoint when an action is **hard to undo, costly if wrong, or touches
something outside the system's own sandbox** — sending real emails, spending money, deleting
data, or publishing content under the company's name; the downside of a bad autonomous call
outweighs the friction of a pause. Full autonomy is reasonable when actions are **cheap,
reversible, and internal** — draft generation, searches, calculations, or writes to a scratch
workspace the agent already controls — where mistakes are low-cost and self-correction loops
(like Task 3's critique loop) are enough of a safety net on their own.


## Task 5 — Persistence & Debugging

The `MemorySaver` checkpointer used above already persists every state transition, keyed by
`thread_id`. That gives us two things for free: (1) we can **resume a paused/finished thread
later** just by invoking again with the same `thread_id`, and (2) we can walk the
**state history** of a thread to replay/debug a run step by step (LangGraph's "time travel").


In [11]:
# (1) Persistence across "sessions": re-open thread-2 later and just read its final state
#     without re-running anything - this simulates restarting the process/kernel.
reopened_state = hitl_app.get_state(thread_2)
print("Re-opened thread-2 final answer:", reopened_state.values["final_answer"])
print("Re-opened thread-2 is finished (no next node):", reopened_state.next == ())


Re-opened thread-2 final answer: [PUBLISHED] Q: What makes LangGraph different from a plain agent loop?
A: LangGraph models agents as explicit graphs of nodes and edges over a shared state object. Compared to a plain loop, this makes branching, cycles, and pausing first-class concepts.
Re-opened thread-2 is finished (no next node): True


In [12]:
# (2) Time travel / debugging: walk the full checkpoint history of thread-2 in order.
print("--- State history for thread-2 (oldest -> newest) ---")
history = list(hitl_app.get_state_history(thread_2))
for snapshot in reversed(history):
    last_node = snapshot.metadata.get("writes")
    step = snapshot.metadata.get("step")
    print(f"step={step:>2}  next={snapshot.next}  log_so_far={snapshot.values.get("log")}")


--- State history for thread-2 (oldest -> newest) ---
step=-1  next=('__start__',)  log_so_far=None
step= 0  next=('plan',)  log_so_far=[]
step= 1  next=('retrieve',)  log_so_far=['plan']
step= 2  next=('generate',)  log_so_far=['plan', 'retrieve']
step= 3  next=('critique',)  log_so_far=['plan', 'retrieve', 'generate']
step= 4  next=('revise',)  log_so_far=['plan', 'retrieve', 'generate', 'critique(score=0.55)']
step= 5  next=('generate',)  log_so_far=['plan', 'retrieve', 'generate', 'critique(score=0.55)', 'revise']
step= 6  next=('critique',)  log_so_far=['plan', 'retrieve', 'generate', 'critique(score=0.55)', 'revise', 'generate']
step= 7  next=('revise',)  log_so_far=['plan', 'retrieve', 'generate', 'critique(score=0.55)', 'revise', 'generate', 'critique(score=0.55)']
step= 8  next=('generate',)  log_so_far=['plan', 'retrieve', 'generate', 'critique(score=0.55)', 'revise', 'generate', 'critique(score=0.55)', 'revise']
step= 9  next=('critique',)  log_so_far=['plan', 'retrieve', 'g

In [13]:
# Replay: re-run the graph starting from an earlier checkpoint (e.g. right after "retrieve")
# by grabbing that snapshot's config and invoking from there - this is the "time travel" trick.
retrieve_checkpoint = next(s for s in history if s.values.get("log", [])[-1:] == ["retrieve"])
print("Replaying from the checkpoint right after `retrieve`...")
replayed = hitl_app.invoke(None, config=retrieve_checkpoint.config)
print("Replay result trace:", replayed.get("log"))


Replaying from the checkpoint right after `retrieve`...
Replay result trace: ['plan', 'retrieve', 'generate', 'critique(score=0.55)', 'revise', 'generate', 'critique(score=0.55)', 'revise', 'generate', 'critique(score=0.55)', 'human_review (awaiting approval)']


**`AgentExecutor` vs. `LangGraph` — when to reach for each**

| | LangChain `AgentExecutor` | LangGraph |
|---|---|---|
| Control flow | Implicit, LLM-driven `think/act/observe` loop | Explicit graph of nodes/edges you design |
| Branching & cycles | Awkward — only what the model's own reasoning produces | First-class via conditional edges, including loops |
| Pausing / human approval | Not built in; you'd bolt it on manually | Native via `interrupt_before` / `interrupt_after` |
| Persistence & replay | Not built in | Native via checkpointers + state history |
| Best for | Quick single-shot tool-using agents, prototypes, simple Q&A-with-tools | Multi-step workflows that branch, self-correct, need approval gates, or must survive restarts |

In practice: reach for `AgentExecutor` when the task is "call some tools until you have an
answer" and you don't need to control *how* it loops. Reach for LangGraph as soon as the
workflow has more than one logical stage, needs a safety loop (like the critique cycle here),
needs a human checkpoint before a risky step, or needs to be pausable/resumable/debuggable as a
long-running process.


## Graph Diagram

Mermaid source for the final workflow (renders directly on GitHub/most Markdown viewers), plus an attempt to auto-generate it straight from the compiled graph so it always matches the code above.

In [14]:
mermaid_source = hitl_app.get_graph().draw_mermaid()
print(mermaid_source)


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	plan(plan)
	retrieve(retrieve)
	generate(generate)
	critique(critique)
	revise(revise)
	human_review(human_review)
	publish(publish<hr/><small><em>__interrupt = before</em></small>)
	reject(reject<hr/><small><em>__interrupt = before</em></small>)
	__end__([<p>__end__</p>]):::last
	__start__ --> plan;
	critique -. &nbsp;finish&nbsp; .-> human_review;
	critique -. &nbsp;loop_back&nbsp; .-> revise;
	generate --> critique;
	human_review -.-> publish;
	human_review -.-> reject;
	plan --> retrieve;
	retrieve --> generate;
	revise --> generate;
	publish --> __end__;
	reject --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [ ]:
# Best-effort PNG render of the graph (skips silently if the optional mermaid renderer
# dependency / network access isn't available in this environment).
try:
    png_bytes = hitl_app.get_graph().draw_mermaid_png()
    with open("langgraph_workflow.png", "wb") as f:
        f.write(png_bytes)
    print("Saved diagram to langgraph_workflow.png")
except Exception as e:
    print(f"PNG render skipped ({e}); use the Mermaid source printed above instead.")


```mermaid
graph TD
    START((START)) --> plan
    plan --> retrieve
    retrieve --> generate
    generate --> critique
    critique -- loop_back --> revise
    revise --> generate
    critique -- finish --> human_review
    human_review -- approve --> publish
    human_review -- reject --> reject
    publish --> END((END))
    reject --> END
```


## Summary

- **Task 1:** defined `StateGraph`/nodes/edges/conditional edges/state, designed an
  `AgentState` schema, and sketched the target graph before writing code.
- **Task 2:** built and ran a linear `plan -> retrieve -> generate -> format` graph, printing
  state after every node.
- **Task 3:** turned `generate -> critique` into a bounded self-correction loop via a
  conditional edge and a `revision_count` guard, and explained why this is natural in LangGraph.
- **Task 4:** added an `interrupt_before` checkpoint in front of the risky `publish` action and
  demonstrated resuming after both a simulated rejection and a simulated approval.
- **Task 5:** used the `MemorySaver` checkpointer to persist and re-open a thread's state, and
  used `get_state_history` to time-travel/replay a run from an earlier checkpoint.
